# Download dataset

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import datetime

def setup_folders():
    """Create input folder and current year subfolder"""
    input_folder = Path("input")
    input_folder.mkdir(exist_ok=True)

    current_year = datetime.datetime.now().year
    current_year_folder = input_folder / str(current_year)
    current_year_folder.mkdir(exist_ok=True)

    return input_folder, current_year_folder

def download_file(url, folder, filename=None):
    """Download a file from URL to specified folder"""
    if filename is None:
        filename = url.split('/')[-1]

    filepath = folder / filename

    # Skip if file already exists
    if filepath.exists():
        print(f"File already exists: {filepath}")
        return True

    try:
        print(f"Downloading: {filename}")
        response = requests.get(url, stream=True)
        response.raise_for_status()

        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"Successfully downloaded: {filepath}")
        return True

    except requests.exceptions.RequestException as e:
        print(f"Error downloading {filename}: {e}")
        return False

def get_download_links(soup, base_url, current_year):
    """Extract all download links from the page"""
    weekly_links = []
    annual_links = []

    # Get weekly links (current year)
    weekly_buttons = soup.select('div.weekly a.btn-sales-data')
    for button in weekly_buttons:
        href = button.get('href')
        if href and str(current_year) in href:
            full_url = urljoin(base_url, href)
            weekly_links.append(full_url)

    # Get annual links (all other years)
    annual_buttons = soup.select('div.annual a.btn-sales-data')
    for button in annual_buttons:
        href = button.get('href')
        if href:
            full_url = urljoin(base_url, href)
            annual_links.append(full_url)

    return weekly_links, annual_links

def main():
    # Configuration
    target_url = "https://valuation.property.nsw.gov.au/embed/propertySalesInformation"
    current_year = datetime.datetime.now().year

    print(f"Starting download...")
    print(f"Current year: {current_year}")

    try:
        # Fetch the webpage
        response = requests.get(target_url)
        response.raise_for_status()

        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Create folders
        input_folder, current_year_folder = setup_folders()
        print(f"Created folder structure:")
        print(f"  - {input_folder}/")
        print(f"  - {current_year_folder}/")

        # Get all download links
        weekly_links, annual_links = get_download_links(soup, target_url, current_year)

        print(f"\nFound {len(weekly_links)} weekly files for {current_year}")
        print(f"Found {len(annual_links)} annual files for other years")

        # Download annual files (direct to input folder)
        print(f"\nDownloading annual files...")
        successful_annual = 0
        for link in annual_links:
            if download_file(link, input_folder):
                successful_annual += 1

        # Download weekly files (to current year folder)
        print(f"\nDownloading weekly files for {current_year}...")
        successful_weekly = 0
        for link in weekly_links:
            if download_file(link, current_year_folder):
                successful_weekly += 1

        # Summary
        print(f"\n=== Download Summary ===")
        print(f"Annual files: {successful_annual}/{len(annual_links)}")
        print(f"Weekly files for {current_year}: {successful_weekly}/{len(weekly_links)}")
        print(f"All files saved in: {input_folder}/")

    except requests.exceptions.RequestException as e:
        print(f"Error accessing website: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()

Starting download...
Current year: 2025
Created folder structure:
  - input/
  - input/2025/

Found 49 weekly files for 2025
Found 35 annual files for other years

Downloading: 1990.zip
Successfully downloaded: input/1990.zip
Downloading: 1991.zip
Successfully downloaded: input/1991.zip
Downloading: 1992.zip
Successfully downloaded: input/1992.zip
Downloading: 1993.zip
Successfully downloaded: input/1993.zip
Downloading: 1994.zip
Successfully downloaded: input/1994.zip
Downloading: 1995.zip
Successfully downloaded: input/1995.zip
Downloading: 1996.zip
Successfully downloaded: input/1996.zip
Downloading: 1997.zip
Successfully downloaded: input/1997.zip
Downloading: 1998.zip
Successfully downloaded: input/1998.zip
Downloading: 1999.zip
Successfully downloaded: input/1999.zip
Downloading: 2000.zip
Successfully downloaded: input/2000.zip
Downloading: 2001.zip
Successfully downloaded: input/2001.zip
Downloading: 2002.zip
Successfully downloaded: input/2002.zip
Downloading: 2003.zip
Successf

# Three scripts archived, current and 2025

In [ ]:
"""
Script 1: Process ARCHIVED Data (1990-2000) [SELF-CONTAINED FOR COLAB]

This script is standalone. It includes all necessary code from
parser_library.py to run in a single Colab cell.
"""
import logging
from pathlib import Path
import io, zipfile, re, json
from typing import Optional, Dict, List, Callable
import polars as pl

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output/archived_sales")
# ---------------------

# ###########################################################################
# ### START: PASTE-IN FROM parser_library.py
# ###########################################################################

# (2001-Current) [cite: 25]
CURRENT_SALES_SCHEMA_MAP = {
    1: "district_code",
    2: "property_id",
    3: "sale_counter",
    4: "download_datetime",
    5: "property_name",
    6: "unit_number",
    7: "house_number",
    8: "street_name",
    9: "locality",
    10: "postcode",
    11: "area",
    12: "area_type",
    13: "contract_date",
    14: "settlement_date",
    15: "purchase_price",
    16: "zoning",
    17: "nature_of_property",
    18: "primary_purpose",
    19: "strata_lot_number",
    20: "component_code",
    21: "sale_code",
    22: "percent_interest_of_sale",
    23: "dealing_number",
}

# (1990-2001) [cite: 31-32]
ARCHIVED_SALES_SCHEMA_MAP = {
    1: "district_code",
    2: "source",
    3: "valuation_num",
    4: "property_id",
    5: "unit_number",
    6: "house_number",
    7: "street_name",
    8: "suburb_name", # We keep this name! dbt will rename it.
    9: "postcode",
    10: "contract_date",
    11: "purchase_price",
    12: "legal_description",
    13: "area",
    14: "area_type",
    15: "dimensions",
    16: "component_code",
    17: "zoning",
}

# --- Polars schemas for writing Parquet files ---
CURRENT_SCHEMA_POLARS = {
    "source_file": pl.Utf8,
    "legal_description": pl.Utf8,
    "district_code": pl.Utf8,
    "property_id": pl.Utf8,
    "sale_counter": pl.Utf8,
    "download_datetime": pl.Utf8,
    "property_name": pl.Utf8,
    "unit_number": pl.Utf8,
    "house_number": pl.Utf8,
    "street_name": pl.Utf8,
    "locality": pl.Utf8,
    "postcode": pl.Utf8,
    "area": pl.Utf8,
    "area_type": pl.Utf8,
    "contract_date": pl.Utf8,
    "settlement_date": pl.Utf8,
    "purchase_price": pl.Utf8,
    "zoning": pl.Utf8,
    "nature_of_property": pl.Utf8,
    "primary_purpose": pl.Utf8,
    "strata_lot_number": pl.Utf8,
    "component_code": pl.Utf8,
    "sale_code": pl.Utf8,
    "percent_interest_of_sale": pl.Utf8,
    "dealing_number": pl.Utf8,
}

ARCHIVED_SCHEMA_POLARS = {
    "source_file": pl.Utf8,
    "district_code": pl.Utf8,
    "source": pl.Utf8,
    "valuation_num": pl.Utf8,
    "property_id": pl.Utf8,
    "unit_number": pl.Utf8,
    "house_number": pl.Utf8,
    "street_name": pl.Utf8,
    "suburb_name": pl.Utf8, # Kept as-is
    "postcode": pl.Utf8,
    "contract_date": pl.Utf8,
    "purchase_price": pl.Utf8,
    "legal_description": pl.Utf8,
    "area": pl.Utf8,
    "area_type": pl.Utf8,
    "dimensions": pl.Utf8,
    "component_code": pl.Utf8,
    "zoning": pl.Utf8,
}

# --- Helper Functions ---

YEAR_RE = re.compile(r'\b(199\d|20\d{2})\b')
YMD8_RE = re.compile(r'^\d{8}$')
DMY_RE = re.compile(r'^\d{1,2}/\d{1,2}/\d{4}$')

def extract_year_from_filename(name: str) -> Optional[int]:
    """Finds the first 4-digit year (199x or 20xx) in a filename."""
    m = YEAR_RE.search(name)
    return int(m.group(0)) if m else None

def try_normalize_date_str(s: Optional[str]) -> Optional[str]:
    """Tries to convert various date formats to ISO YYYY-MM-DD."""
    if not s:
        return None
    s = s.strip()
    if YMD8_RE.match(s):
        # Format CCYYMMDD
        return f"{s[0:4]}-{s[4:6]}-{s[6:8]}"
    if DMY_RE.match(s):
        # Format D/M/YYYY or DD/MM/YYYY
        d,m,y = s.split("/")
        return f"{int(y):04d}-{int(m):02d}-{int(d):02d}"
    if re.match(r'^\d{4}-\d{2}-\d{2}', s):
        # Already in ISO format (e.g., from a T-timestamp)
        return s.split("T")[0]
    return None

def split_semicolon_line(line: str) -> List[str]:
    """Splits a line from the .dat file by semicolon."""
    return line.rstrip("\n").split(";")

# --- Core Parsing Functions ---

def parse_archived_dat_stream(stream: io.TextIOBase, src_label: str) -> List[Dict]:
    """
    Parses a .dat file stream using the simple ARCHIVED format.
    This format only has 'B' records for sales.
    """
    batch = []
    line_no = 0
    for raw in stream:
        line_no += 1
        if not raw.strip():
            continue
        try:
            parts = split_semicolon_line(raw)
            rec_type = parts[0].strip() if parts else ""

            if rec_type == "B":
                rec = {"source_file": src_label}
                for idx, colname in ARCHIVED_SALES_SCHEMA_MAP.items():
                    val = parts[idx] if idx < len(parts) else ""
                    rec[colname] = val
                batch.append(rec)
        except Exception as e:
            logging.warning(f"Bad row in {src_label} (line {line_no}): {e}")

    return batch

def parse_current_dat_stream(stream: io.TextIOBase, src_label: str) -> List[Dict]:
    """
    Parses a .dat file stream using the CURRENT format.
    This format has 'B' records for sales and 'C' records
    for legal descriptions.
    """
    b_records_in_progress = {} # To join C records
    final_batch = []
    line_no = 0

    for raw in stream:
        line_no += 1
        if not raw.strip():
            continue
        try:
            parts = split_semicolon_line(raw)
            rec_type = parts[0].strip() if parts else ""

            if rec_type == "B":
                rec = {"source_file": src_label}
                for idx, colname in CURRENT_SALES_SCHEMA_MAP.items():
                    val = parts[idx] if idx < len(parts) else ""
                    rec[colname] = val

                # Pre-allocate legal_description
                rec["legal_description"] = ""

                # Create a unique key to find this B record later
                key = (rec.get("district_code"), rec.get("property_id"), rec.get("sale_counter"))
                b_records_in_progress[key] = rec
                final_batch.append(rec)

            elif rec_type == "C":
                if len(parts) >= 6:
                    # Key: district, property_id, sale_counter
                    key = (parts[1], parts[2], parts[3])
                    legal_part = parts[5] # Property Legal Description

                    # Find the matching B record and append the description
                    if key in b_records_in_progress:
                        b_records_in_progress[key]["legal_description"] += (legal_part or "")

        except Exception as e:
            logging.warning(f"Bad row in {src_label} (line {line_no}): {e}")

    return final_batch

def process_zip_file(
    zip_path: Path,
    parser_function: Callable[[io.TextIOBase, str], List[Dict]]
) -> List[Dict]:
    """
    A generic function to process a single ZIP file.
    It will:
    1. Open the zip file.
    2. Find all .dat files inside (even in nested zips).
    3. Use the provided `parser_function` to parse them.
    Returns a single list of all records found.
    """
    all_records = []
    try:
        # zip_path can be a Path object or a file-like object (for nested zips)
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.infolist():
                if member.is_dir():
                    continue

                name = member.filename

                # Determine src_label based on if zip_path is Path or not
                if isinstance(zip_path, Path):
                    src_label = f"{zip_path.name}/{name}"
                else:
                    src_label = f"nested_zip/{name}" # Handle nested

                try:
                    with zf.open(member) as fh:
                        if name.lower().endswith(".zip"):
                            # Handle nested zips
                            logging.info(f"  -> Found nested zip: {name}")
                            nested_bytes = io.BytesIO(fh.read())
                            # Recursively call this function, passing the parser
                            all_records.extend(process_zip_file(nested_bytes, parser_function))

                        elif name.lower().endswith(".dat"):
                            # Process .dat file
                            logging.info(f"  -> Parsing dat: {name}")
                            raw_bytes = fh.read()
                            txt_stream = io.TextIOWrapper(io.BytesIO(raw_bytes), encoding="utf-8", errors="replace")
                            all_records.extend(parser_function(txt_stream, src_label))

                except Exception as e:
                    logging.error(f"Error processing member {name} in {zip_path}: {e}")

    except zipfile.BadZipFile:
        logging.error(f"Bad ZIP file, skipping: {zip_path}")

    return all_records

def save_to_parquet(records: List[Dict], schema: Dict, output_path: Path, year: int):
    """
    Converts a list of records to a Polars DataFrame and saves as Parquet.
    It saves into a dbt/Athena-friendly partition: `output_path/year=YYYY/sales.parquet`
    """
    if not records:
        logging.warning(f"No records found for year {year}. Skipping.")
        return

    try:
        # Convert list of dictionaries to Polars DataFrame
        df = pl.from_dicts(records, schema=schema)

        # Clean date columns
        if "contract_date" in df.columns:
            # FIX: Use map_elements (replaces apply) for Polars expressions
            df = df.with_columns(
                pl.col("contract_date").map_elements(try_normalize_date_str, return_dtype=pl.Utf8).alias("contract_date")
            )
        if "settlement_date" in df.columns:
            # FIX: Use map_elements (replaces apply) for Polars expressions
            df = df.with_columns(
                pl.col("settlement_date").map_elements(try_normalize_date_str, return_dtype=pl.Utf8).alias("settlement_date")
            )

        # # Add the year column for partitioning
        # df = df.with_columns(pl.lit(year).alias("year"))

        # Define the final partitioned output path
        partition_dir = output_path / f"year={year}"
        partition_dir.mkdir(parents=True, exist_ok=True)
        final_file = partition_dir / "sales.parquet"

        df.write_parquet(final_file, compression="snappy")
        logging.info(f"SUCCESS: Saved {len(df)} rows for year {year} to {final_file}")

    except Exception as e:
        logging.error(f"Failed to save Parquet for year {year}: {e}")

# ###########################################################################
# ### END: PASTE-IN FROM parser_library.py
# ###########################################################################


def main():
    logging.info("--- STARTING SCRIPT 1: ARCHIVED (1990-2000) ---")

    if not INPUT_DIR.exists():
        logging.warning(f"Input directory '{INPUT_DIR}' not found. Exiting.")
        return

    # Get all .zip files in the main input folder
    all_zips = list(INPUT_DIR.glob("*.zip"))

    if not all_zips:
        logging.warning(f"No .zip files found in {INPUT_DIR}. Exiting.")
        return

    logging.info(f"Found {len(all_zips)} total zip files. Checking for archived data...")

    # We will process one year at a time
    processed_years = []

    for zip_path in all_zips:
        year = extract_year_from_filename(zip_path.name)

        if not year:
            logging.warning(f"Could not find year in {zip_path.name}. Skipping.")
            continue

        # This script ONLY processes ARCHIVED years (<= 2000)
        if year > 2000:
            continue

        logging.info(f"Processing ARCHIVED year {year} from {zip_path.name}...")

        # 1. Parse all .dat files (including nested zips)
        records = process_zip_file(
            zip_path=zip_path,
            parser_function=parse_archived_dat_stream # <-- Tell it to use the ARCHIVED parser
        )

        # 2. Save the results to a partitioned Parquet file
        save_to_parquet(
            records=records,
            schema=ARCHIVED_SCHEMA_POLARS, # <-- Tell it to use the ARCHIVED schema
            output_path=OUTPUT_DIR,
            year=year
        )
        processed_years.append(year)

    logging.info("--- SCRIPT 1: FINISHED ---")
    if processed_years:
        logging.info(f"Successfully processed archived years: {sorted(list(set(processed_years)))}")
    else:
        logging.info("No new archived data was found to process.")

if __name__ == "__main__":
    # This will run when you paste it in a Colab cell
    main()



In [ ]:
"""
Script 2: Process CURRENT ANNUAL Data (2001-2024) [SELF-CONTAINED FOR COLAB]

This script is standalone. It includes all necessary code from
parser_library.py to run in a single Colab cell.
"""

import logging
from pathlib import Path
import io, zipfile, re, json
from typing import Optional, Dict, List, Callable
import polars as pl

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output/historical_annual_sales")
CURRENT_YEAR = 2025 # The year of the weekly data
# ---------------------

# ###########################################################################
# ### START: PASTE-IN FROM parser_library.py
# ###########################################################################

# (2001-Current) [cite: 25]
CURRENT_SALES_SCHEMA_MAP = {
    1: "district_code",
    2: "property_id",
    3: "sale_counter",
    4: "download_datetime",
    5: "property_name",
    6: "unit_number",
    7: "house_number",
    8: "street_name",
    9: "locality",
    10: "postcode",
    11: "area",
    12: "area_type",
    13: "contract_date",
    14: "settlement_date",
    15: "purchase_price",
    16: "zoning",
    17: "nature_of_property",
    18: "primary_purpose",
    19: "strata_lot_number",
    20: "component_code",
    21: "sale_code",
    22: "percent_interest_of_sale",
    23: "dealing_number",
}

# (1990-2001) [cite: 31-32]
ARCHIVED_SALES_SCHEMA_MAP = {
    1: "district_code",
    2: "source",
    3: "valuation_num",
    4: "property_id",
    5: "unit_number",
    6: "house_number",
    7: "street_name",
    8: "suburb_name", # We keep this name! dbt will rename it.
    9: "postcode",
    10: "contract_date",
    11: "purchase_price",
    12: "legal_description",
    13: "area",
    14: "area_type",
    15: "dimensions",
    16: "component_code",
    17: "zoning",
}

# --- Polars schemas for writing Parquet files ---
CURRENT_SCHEMA_POLARS = {
    "source_file": pl.Utf8,
    "legal_description": pl.Utf8,
    "district_code": pl.Utf8,
    "property_id": pl.Utf8,
    "sale_counter": pl.Utf8,
    "download_datetime": pl.Utf8,
    "property_name": pl.Utf8,
    "unit_number": pl.Utf8,
    "house_number": pl.Utf8,
    "street_name": pl.Utf8,
    "locality": pl.Utf8,
    "postcode": pl.Utf8,
    "area": pl.Utf8,
    "area_type": pl.Utf8,
    "contract_date": pl.Utf8,
    "settlement_date": pl.Utf8,
    "purchase_price": pl.Utf8,
    "zoning": pl.Utf8,
    "nature_of_property": pl.Utf8,
    "primary_purpose": pl.Utf8,
    "strata_lot_number": pl.Utf8,
    "component_code": pl.Utf8,
    "sale_code": pl.Utf8,
    "percent_interest_of_sale": pl.Utf8,
    "dealing_number": pl.Utf8,
}

ARCHIVED_SCHEMA_POLARS = {
    "source_file": pl.Utf8,
    "district_code": pl.Utf8,
    "source": pl.Utf8,
    "valuation_num": pl.Utf8,
    "property_id": pl.Utf8,
    "unit_number": pl.Utf8,
    "house_number": pl.Utf8,
    "street_name": pl.Utf8,
    "suburb_name": pl.Utf8, # Kept as-is
    "postcode": pl.Utf8,
    "contract_date": pl.Utf8,
    "purchase_price": pl.Utf8,
    "legal_description": pl.Utf8,
    "area": pl.Utf8,
    "area_type": pl.Utf8,
    "dimensions": pl.Utf8,
    "component_code": pl.Utf8,
    "zoning": pl.Utf8,
}

# --- Helper Functions ---

YEAR_RE = re.compile(r'\b(199\d|20\d{2})\b')
YMD8_RE = re.compile(r'^\d{8}$')
DMY_RE = re.compile(r'^\d{1,2}/\d{1,2}/\d{4}$')

def extract_year_from_filename(name: str) -> Optional[int]:
    """Finds the first 4-digit year (199x or 20xx) in a filename."""
    m = YEAR_RE.search(name)
    return int(m.group(0)) if m else None

def try_normalize_date_str(s: Optional[str]) -> Optional[str]:
    """Tries to convert various date formats to ISO YYYY-MM-DD."""
    if not s:
        return None
    s = s.strip()
    if YMD8_RE.match(s):
        # Format CCYYMMDD
        return f"{s[0:4]}-{s[4:6]}-{s[6:8]}"
    if DMY_RE.match(s):
        # Format D/M/YYYY or DD/MM/YYYY
        d,m,y = s.split("/")
        return f"{int(y):04d}-{int(m):02d}-{int(d):02d}"
    if re.match(r'^\d{4}-\d{2}-\d{2}', s):
        # Already in ISO format (e.g., from a T-timestamp)
        return s.split("T")[0]
    return None

def split_semicolon_line(line: str) -> List[str]:
    """Splits a line from the .dat file by semicolon."""
    return line.rstrip("\n").split(";")

# --- Core Parsing Functions ---

def parse_archived_dat_stream(stream: io.TextIOBase, src_label: str) -> List[Dict]:
    """
    Parses a .dat file stream using the simple ARCHIVED format.
    This format only has 'B' records for sales.
    """
    batch = []
    line_no = 0
    for raw in stream:
        line_no += 1
        if not raw.strip():
            continue
        try:
            parts = split_semicolon_line(raw)
            rec_type = parts[0].strip() if parts else ""

            if rec_type == "B":
                rec = {"source_file": src_label}
                for idx, colname in ARCHIVED_SALES_SCHEMA_MAP.items():
                    val = parts[idx] if idx < len(parts) else ""
                    rec[colname] = val
                batch.append(rec)
        except Exception as e:
            logging.warning(f"Bad row in {src_label} (line {line_no}): {e}")

    return batch

def parse_current_dat_stream(stream: io.TextIOBase, src_label: str) -> List[Dict]:
    """
    Parses a .dat file stream using the CURRENT format.
    This format has 'B' records for sales and 'C' records
    for legal descriptions.
    """
    b_records_in_progress = {} # To join C records
    final_batch = []
    line_no = 0

    for raw in stream:
        line_no += 1
        if not raw.strip():
            continue
        try:
            parts = split_semicolon_line(raw)
            rec_type = parts[0].strip() if parts else ""

            if rec_type == "B":
                rec = {"source_file": src_label}
                for idx, colname in CURRENT_SALES_SCHEMA_MAP.items():
                    val = parts[idx] if idx < len(parts) else ""
                    rec[colname] = val

                # Pre-allocate legal_description
                rec["legal_description"] = ""

                # Create a unique key to find this B record later
                key = (rec.get("district_code"), rec.get("property_id"), rec.get("sale_counter"))
                b_records_in_progress[key] = rec
                final_batch.append(rec)

            elif rec_type == "C":
                if len(parts) >= 6:
                    # Key: district, property_id, sale_counter
                    key = (parts[1], parts[2], parts[3])
                    legal_part = parts[5] # Property Legal Description

                    # Find the matching B record and append the description
                    if key in b_records_in_progress:
                        b_records_in_progress[key]["legal_description"] += (legal_part or "")

        except Exception as e:
            logging.warning(f"Bad row in {src_label} (line {line_no}): {e}")

    return final_batch

def process_zip_file(
    zip_path: Path,
    parser_function: Callable[[io.TextIOBase, str], List[Dict]]
) -> List[Dict]:
    """
    A generic function to process a single ZIP file.
    It will:
    1. Open the zip file.
    2. Find all .dat files inside (even in nested zips).
    3. Use the provided `parser_function` to parse them.
    Returns a single list of all records found.
    """
    all_records = []
    try:
        # zip_path can be a Path object or a file-like object (for nested zips)
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.infolist():
                if member.is_dir():
                    continue

                name = member.filename

                # Determine src_label based on if zip_path is Path or not
                if isinstance(zip_path, Path):
                    src_label = f"{zip_path.name}/{name}"
                else:
                    src_label = f"nested_zip/{name}" # Handle nested

                try:
                    with zf.open(member) as fh:
                        if name.lower().endswith(".zip"):
                            # Handle nested zips
                            logging.info(f"  -> Found nested zip: {name}")
                            nested_bytes = io.BytesIO(fh.read())
                            # Recursively call this function, passing the parser
                            all_records.extend(process_zip_file(nested_bytes, parser_function))

                        elif name.lower().endswith(".dat"):
                            # Process .dat file
                            logging.info(f"  -> Parsing dat: {name}")
                            raw_bytes = fh.read()
                            txt_stream = io.TextIOWrapper(io.BytesIO(raw_bytes), encoding="utf-8", errors="replace")
                            all_records.extend(parser_function(txt_stream, src_label))

                except Exception as e:
                    logging.error(f"Error processing member {name} in {zip_path}: {e}")

    except zipfile.BadZipFile:
        logging.error(f"Bad ZIP file, skipping: {zip_path}")

    return all_records

def save_to_parquet(records: List[Dict], schema: Dict, output_path: Path, year: int):
    """
    Converts a list of records to a Polars DataFrame and saves as Parquet.
    It saves into a dbt/Athena-friendly partition: `output_path/year=YYYY/sales.parquet`
    """
    if not records:
        logging.warning(f"No records found for year {year}. Skipping.")
        return

    try:
        # Convert list of dictionaries to Polars DataFrame
        df = pl.from_dicts(records, schema=schema)

        # Clean date columns
        if "contract_date" in df.columns:
            # FIX: Use map_elements (replaces apply) for Polars expressions
            df = df.with_columns(
                pl.col("contract_date").map_elements(try_normalize_date_str, return_dtype=pl.Utf8).alias("contract_date")
            )
        if "settlement_date" in df.columns:
            # FIX: Use map_elements (replaces apply) for Polars expressions
            df = df.with_columns(
                pl.col("settlement_date").map_elements(try_normalize_date_str, return_dtype=pl.Utf8).alias("settlement_date")
            )

        # # Add the year column for partitioning
        # df = df.with_columns(pl.lit(year).alias("year"))

        # Define the final partitioned output path
        partition_dir = output_path / f"year={year}"
        partition_dir.mkdir(parents=True, exist_ok=True)
        final_file = partition_dir / "sales.parquet"

        df.write_parquet(final_file, compression="snappy")
        logging.info(f"SUCCESS: Saved {len(df)} rows for year {year} to {final_file}")

    except Exception as e:
        logging.error(f"Failed to save Parquet for year {year}: {e}")

# ###########################################################################
# ### END: PASTE-IN FROM parser_library.py
# ###########################################################################


def main():
    logging.info("--- STARTING SCRIPT 2: CURRENT ANNUAL (2001-2024) ---")

    if not INPUT_DIR.exists():
        logging.warning(f"Input directory '{INPUT_DIR}' not found. Exiting.")
        return

    # Get all .zip files in the main input folder
    all_zips = list(INPUT_DIR.glob("*.zip"))

    if not all_zips:
        logging.warning(f"No .zip files found in {INPUT_DIR}. Exiting.")
        return

    logging.info(f"Found {len(all_zips)} total zip files. Checking for current annual data...")

    # We will process one year at a time
    processed_years = []

    for zip_path in all_zips:
        year = extract_year_from_filename(zip_path.name)

        if not year:
            logging.warning(f"Could not find year in {zip_path.name}. Skipping.")
            continue

        # This script ONLY processes CURRENT ANNUAL years (2001-2024)
        if year <= 2000 or year >= CURRENT_YEAR:
            continue

        logging.info(f"Processing CURRENT year {year} from {zip_path.name}...")

        # 1. Parse all .dat files (including nested zips)
        records = process_zip_file(
            zip_path=zip_path,
            parser_function=parse_current_dat_stream # <-- Tell it to use the CURRENT parser
        )

        # 2. Save the results to a partitioned Parquet file
        save_to_parquet(
            records=records,
            schema=CURRENT_SCHEMA_POLARS, # <-- Tell it to use the CURRENT schema
            output_path=OUTPUT_DIR,
            year=year
        )
        processed_years.append(year)

    logging.info("--- SCRIPT 2: FINISHED ---")
    if processed_years:
        logging.info(f"Successfully processed current annual years: {sorted(list(set(processed_years)))}")
    else:
        logging.info("No new current annual data was found to process.")

if __name__ == "__main__":
    # This will run when you paste it in a Colab cell
    main()



In [ ]:
"""
Script 3: Process CURRENT WEEKLY Data (2025) [SELF-CONTAINED FOR COLAB]

This script is standalone. It includes all necessary code from
parser_library.py to run in a single Colab cell.
"""

import logging
from pathlib import Path
import io, zipfile, re, json
from typing import Optional, Dict, List, Callable
import polars as pl

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
CURRENT_YEAR = 2025 # The year of the weekly data
INPUT_DIR = Path(f"input/{CURRENT_YEAR}")
OUTPUT_DIR = Path("output/current_year_weekly_sales")
# ---------------------

# ###########################################################################
# ### START: PASTE-IN FROM parser_library.py
# ###########################################################################

# (2001-Current) [cite: 25]
CURRENT_SALES_SCHEMA_MAP = {
    1: "district_code",
    2: "property_id",
    3: "sale_counter",
    4: "download_datetime",
    5: "property_name",
    6: "unit_number",
    7: "house_number",
    8: "street_name",
    9: "locality",
    10: "postcode",
    11: "area",
    12: "area_type",
    13: "contract_date",
    14: "settlement_date",
    15: "purchase_price",
    16: "zoning",
    17: "nature_of_property",
    18: "primary_purpose",
    19: "strata_lot_number",
    20: "component_code",
    21: "sale_code",
    22: "percent_interest_of_sale",
    23: "dealing_number",
}

# (1990-2001) [cite: 31-32]
ARCHIVED_SALES_SCHEMA_MAP = {
    1: "district_code",
    2: "source",
    3: "valuation_num",
    4: "property_id",
    5: "unit_number",
    6: "house_number",
    7: "street_name",
    8: "suburb_name", # We keep this name! dbt will rename it.
    9: "postcode",
    10: "contract_date",
    11: "purchase_price",
    12: "legal_description",
    13: "area",
    14: "area_type",
    15: "dimensions",
    16: "component_code",
    17: "zoning",
}

# --- Polars schemas for writing Parquet files ---
CURRENT_SCHEMA_POLARS = {
    "source_file": pl.Utf8,
    "legal_description": pl.Utf8,
    "district_code": pl.Utf8,
    "property_id": pl.Utf8,
    "sale_counter": pl.Utf8,
    "download_datetime": pl.Utf8,
    "property_name": pl.Utf8,
    "unit_number": pl.Utf8,
    "house_number": pl.Utf8,
    "street_name": pl.Utf8,
    "locality": pl.Utf8,
    "postcode": pl.Utf8,
    "area": pl.Utf8,
    "area_type": pl.Utf8,
    "contract_date": pl.Utf8,
    "settlement_date": pl.Utf8,
    "purchase_price": pl.Utf8,
    "zoning": pl.Utf8,
    "nature_of_property": pl.Utf8,
    "primary_purpose": pl.Utf8,
    "strata_lot_number": pl.Utf8,
    "component_code": pl.Utf8,
    "sale_code": pl.Utf8,
    "percent_interest_of_sale": pl.Utf8,
    "dealing_number": pl.Utf8,
}

ARCHIVED_SCHEMA_POLARS = {
    "source_file": pl.Utf8,
    "district_code": pl.Utf8,
    "source": pl.Utf8,
    "valuation_num": pl.Utf8,
    "property_id": pl.Utf8,
    "unit_number": pl.Utf8,
    "house_number": pl.Utf8,
    "street_name": pl.Utf8,
    "suburb_name": pl.Utf8, # Kept as-is
    "postcode": pl.Utf8,
    "contract_date": pl.Utf8,
    "purchase_price": pl.Utf8,
    "legal_description": pl.Utf8,
    "area": pl.Utf8,
    "area_type": pl.Utf8,
    "dimensions": pl.Utf8,
    "component_code": pl.Utf8,
    "zoning": pl.Utf8,
}

# --- Helper Functions ---

YEAR_RE = re.compile(r'\b(199\d|20\d{2})\b')
YMD8_RE = re.compile(r'^\d{8}$')
DMY_RE = re.compile(r'^\d{1,2}/\d{1,2}/\d{4}$')

def extract_year_from_filename(name: str) -> Optional[int]:
    """Finds the first 4-digit year (199x or 20xx) in a filename."""
    m = YEAR_RE.search(name)
    return int(m.group(0)) if m else None

def try_normalize_date_str(s: Optional[str]) -> Optional[str]:
    """Tries to convert various date formats to ISO YYYY-MM-DD."""
    if not s:
        return None
    s = s.strip()
    if YMD8_RE.match(s):
        # Format CCYYMMDD
        return f"{s[0:4]}-{s[4:6]}-{s[6:8]}"
    if DMY_RE.match(s):
        # Format D/M/YYYY or DD/MM/YYYY
        d,m,y = s.split("/")
        return f"{int(y):04d}-{int(m):02d}-{int(d):02d}"
    if re.match(r'^\d{4}-\d{2}-\d{2}', s):
        # Already in ISO format (e.g., from a T-timestamp)
        return s.split("T")[0]
    return None

def split_semicolon_line(line: str) -> List[str]:
    """Splits a line from the .dat file by semicolon."""
    return line.rstrip("\n").split(";")

# --- Core Parsing Functions ---

def parse_archived_dat_stream(stream: io.TextIOBase, src_label: str) -> List[Dict]:
    """
    Parses a .dat file stream using the simple ARCHIVED format.
    This format only has 'B' records for sales.
    """
    batch = []
    line_no = 0
    for raw in stream:
        line_no += 1
        if not raw.strip():
            continue
        try:
            parts = split_semicolon_line(raw)
            rec_type = parts[0].strip() if parts else ""

            if rec_type == "B":
                rec = {"source_file": src_label}
                for idx, colname in ARCHIVED_SALES_SCHEMA_MAP.items():
                    val = parts[idx] if idx < len(parts) else ""
                    rec[colname] = val
                batch.append(rec)
        except Exception as e:
            logging.warning(f"Bad row in {src_label} (line {line_no}): {e}")

    return batch

def parse_current_dat_stream(stream: io.TextIOBase, src_label: str) -> List[Dict]:
    """
    Parses a .dat file stream using the CURRENT format.
    This format has 'B' records for sales and 'C' records
    for legal descriptions.
    """
    b_records_in_progress = {} # To join C records
    final_batch = []
    line_no = 0

    for raw in stream:
        line_no += 1
        if not raw.strip():
            continue
        try:
            parts = split_semicolon_line(raw)
            rec_type = parts[0].strip() if parts else ""

            if rec_type == "B":
                rec = {"source_file": src_label}
                for idx, colname in CURRENT_SALES_SCHEMA_MAP.items():
                    val = parts[idx] if idx < len(parts) else ""
                    rec[colname] = val

                # Pre-allocate legal_description
                rec["legal_description"] = ""

                # Create a unique key to find this B record later
                key = (rec.get("district_code"), rec.get("property_id"), rec.get("sale_counter"))
                b_records_in_progress[key] = rec
                final_batch.append(rec)

            elif rec_type == "C":
                if len(parts) >= 6:
                    # Key: district, property_id, sale_counter
                    key = (parts[1], parts[2], parts[3])
                    legal_part = parts[5] # Property Legal Description

                    # Find the matching B record and append the description
                    if key in b_records_in_progress:
                        b_records_in_progress[key]["legal_description"] += (legal_part or "")

        except Exception as e:
            logging.warning(f"Bad row in {src_label} (line {line_no}): {e}")

    return final_batch

def process_zip_file(
    zip_path: Path,
    parser_function: Callable[[io.TextIOBase, str], List[Dict]]
) -> List[Dict]:
    """
    A generic function to process a single ZIP file.
    It will:
    1. Open the zip file.
    2. Find all .dat files inside (even in nested zips).
    3. Use the provided `parser_function` to parse them.
    Returns a single list of all records found.
    """
    all_records = []
    try:
        # zip_path can be a Path object or a file-like object (for nested zips)
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.infolist():
                if member.is_dir():
                    continue

                name = member.filename

                # Determine src_label based on if zip_path is Path or not
                if isinstance(zip_path, Path):
                    src_label = f"{zip_path.name}/{name}"
                else:
                    src_label = f"nested_zip/{name}" # Handle nested

                try:
                    with zf.open(member) as fh:
                        if name.lower().endswith(".zip"):
                            # Handle nested zips
                            logging.info(f"  -> Found nested zip: {name}")
                            nested_bytes = io.BytesIO(fh.read())
                            # Recursively call this function, passing the parser
                            all_records.extend(process_zip_file(nested_bytes, parser_function))

                        elif name.lower().endswith(".dat"):
                            # Process .dat file
                            logging.info(f"  -> Parsing dat: {name}")
                            raw_bytes = fh.read()
                            txt_stream = io.TextIOWrapper(io.BytesIO(raw_bytes), encoding="utf-8", errors="replace")
                            all_records.extend(parser_function(txt_stream, src_label))

                except Exception as e:
                    logging.error(f"Error processing member {name} in {zip_path}: {e}")

    except zipfile.BadZipFile:
        logging.error(f"Bad ZIP file, skipping: {zip_path}")

    return all_records

def save_to_parquet(records: List[Dict], schema: Dict, output_path: Path, year: int):
    """
    Converts a list of records to a Polars DataFrame and saves as Parquet.
    It saves into a dbt/Athena-friendly partition: `output_path/year=YYYY/sales.parquet`
    """
    if not records:
        logging.warning(f"No records found for year {year}. Skipping.")
        return

    try:
        # Convert list of dictionaries to Polars DataFrame
        df = pl.from_dicts(records, schema=schema)

        # Clean date columns
        if "contract_date" in df.columns:
            # FIX: Use map_elements (replaces apply) for Polars expressions
            df = df.with_columns(
                pl.col("contract_date").map_elements(try_normalize_date_str, return_dtype=pl.Utf8).alias("contract_date")
            )
        if "settlement_date" in df.columns:
            # FIX: Use map_elements (replaces apply) for Polars expressions
            df = df.with_columns(
                pl.col("settlement_date").map_elements(try_normalize_date_str, return_dtype=pl.Utf8).alias("settlement_date")
            )

        # # Add the year column for partitioning
        # df = df.with_columns(pl.lit(year).alias("year"))

        # Define the final partitioned output path
        partition_dir = output_path / f"year={year}"
        partition_dir.mkdir(parents=True, exist_ok=True)
        final_file = partition_dir / "sales.parquet"

        df.write_parquet(final_file, compression="snappy")
        logging.info(f"SUCCESS: Saved {len(df)} rows for year {year} to {final_file}")

    except Exception as e:
        logging.error(f"Failed to save Parquet for year {year}: {e}")

# ###########################################################################
# ### END: PASTE-IN FROM parser_library.py
# ###########################################################################


def main():
    logging.info(f"--- STARTING SCRIPT 3: CURRENT WEEKLY ({CURRENT_YEAR}) ---")

    if not INPUT_DIR.exists():
        logging.error(f"Input directory {INPUT_DIR} not found. Did you run the downloader?")
        return

    all_zips = list(INPUT_DIR.glob("*.zip"))

    if not all_zips:
        logging.warning(f"No .zip files found in {INPUT_DIR}. Exiting.")
        return

    logging.info(f"Found {len(all_zips)} weekly zip files to process for {CURRENT_YEAR}...")

    all_records_for_year = []

    for zip_path in all_zips:
        logging.info(f"Processing weekly file {zip_path.name}...")

        # 1. Parse all .dat files (including nested zips)
        records = process_zip_file(
            zip_path=zip_path,
            parser_function=parse_current_dat_stream # <-- Tell it to use the CURRENT parser
        )
        all_records_for_year.extend(records)

    logging.info(f"All weekly files processed. Total records: {len(all_records_for_year)}")

    # 2. Save ALL records to a single partitioned Parquet file
    save_to_parquet(
        records=all_records_for_year,
        schema=CURRENT_SCHEMA_POLARS, # <-- Tell it to use the CURRENT schema
        output_path=OUTPUT_DIR,
        year=CURRENT_YEAR
    )

    logging.info("--- SCRIPT 3: FINISHED ---")

if __name__ == "__main__":
    # This will run when you paste it in a Colab cell
    main()



In [ ]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00


In [ ]:
import os
import re
from pathlib import Path
import logging
import boto3
from botocore.exceptions import NoCredentialsError, ClientError, CredentialRetrievalError
from datetime import datetime
from dotenv import load_dotenv # Keep this for local testing if needed
from google.colab import userdata # Import userdata to access secrets

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables from a .env file if it exists (for local testing)
load_dotenv()

# Define the local directory to upload from and the target S3 prefix
LOCAL_UPLOAD_ROOT = Path("/content/output")
S3_PROCESSED_PREFIX = "processed/nsw_bulk_property_sales"

# Read S3 bucket name and AWS region from Colab Secrets or environment variables
S3_BUCKET_NAME = "au-real-estate" # Replace with your bucket name
AWS_REGION = "us-east-1" # Replace with your desired AWS region


# Read AWS credentials from Colab Secrets or environment variables
AWS_ACCESS_KEY_ID = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')


def upload_directory_to_s3(local_dir: Path, s3_prefix: str, bucket_name: str, s3_client):
    """
    Recursively uploads all files and subdirectories from a local directory to S3.
    """
    if not local_dir.is_dir():
        logger.error(f"Local directory not found: {local_dir}")
        return

    for item_path in local_dir.rglob("*"):
        if item_path.is_file():
            # Construct the S3 key based on the relative path from LOCAL_UPLOAD_ROOT
            relative_path = item_path.relative_to(LOCAL_UPLOAD_ROOT)
            s3_key = f"{s3_prefix}/{relative_path}"

            try:
                logger.info(f"Uploading {item_path} to s3://{bucket_name}/{s3_key}")
                s3_client.upload_file(str(item_path), bucket_name, s3_key)
                logger.info(f"Successfully uploaded {s3_key}")
            except (NoCredentialsError, ClientError) as e:
                logger.error(f"Failed to upload {item_path}: {e}")
            except Exception as e:
                 logger.error(f"An unexpected error occurred during upload of {item_path}: {e}")


def main():
    if not S3_BUCKET_NAME:
        raise CredentialRetrievalError(provider='env/secrets',
                                       error_msg="S3_BUCKET_NAME not found in environment variables or Colab secrets.")

    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
         raise CredentialRetrievalError(provider='env/secrets',
                                       error_msg="AWS_ACCESS_KEY_ID or AWS_SECRET_ACCESS_KEY not found in environment variables or Colab secrets.")

    try:
        # Configure boto3 to use the credentials obtained from secrets/env vars
        s3_client = boto3.client(
            's3',
            region_name=AWS_REGION,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY
        )
        upload_directory_to_s3(LOCAL_UPLOAD_ROOT, S3_PROCESSED_PREFIX, S3_BUCKET_NAME, s3_client)
        logger.info(f"\n=== Upload Complete ===")
        logger.info(f"All files from {LOCAL_UPLOAD_ROOT} uploaded to s3://{S3_BUCKET_NAME}/{S3_PROCESSED_PREFIX}/")
        logger.info("========================")

    except CredentialRetrievalError as e:
        logger.error(f"AWS credentials error: {e}. Please ensure AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, S3_BUCKET_NAME, and optionally AWS_REGION are set as environment variables or in your Colab secrets.")
    except Exception as e:
        logger.error(f"An unexpected error occurred during the S3 upload process: {e}")


if __name__ == "__main__":
    main()